### Middlewares

Middleware provides a way to more tightly control what happens inside the agents. Middleware is useful for the following:

- Tracking agent behaviour with logging, analytics, and debugging.
- Transforming prompts, tools selection and output formatting.
- Adding retries, fallback, and early termination logic.
- Applying rate limits, guardrails, and PIL detection.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

#### Summarization Middleware

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message based summarization

#### Here what this checkpoiter do is it stored the whole conversation in the memory (RAM), and associate it with a thread_id; who whenever the model invokes, it looks for any conversation under that thread_id and load it and send it along with the request to the LLM model.
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages", 10),
            keep=("messages", 4)
        ),
    ]
)

In [3]:
### Thread_id for the checkpointer
config = {"configurable": {"thread_id": "test_1"}}

In [ ]:
### Test data to create the long coversation history
questions = [
    "What is 2*2?",
    "What is 99/34?",
    "What is 3^5?",
    "What is 26199 + 6000?",
    "What is 45-32?",
    "What is 11*11?",
    "What is 144/12?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Message's length: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2*2?', additional_kwargs={}, response_metadata={}, id='76426544-aef5-437f-923a-e6437935abe2'), AIMessage(content='2 * 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_795bd00303', 'id': 'chatcmpl-Dvgb58S4pWkiGsT4nB0oaCknSn2dm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f0dac-1303-7931-84da-22efad20c5b6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': 

In [ ]:
### Token based summarization
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels in a particular city - returns long response inorder to use more token."""
    return f"""
        Hotels in {city}:
        1. Swastik Grand - 5 Star - spa, pool, gym
        2. Arihant Palace - 5 Star - pool, multicuisine, bar
        3. Navneeta - 3 Star - pool, reception
    """

agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        ),
    ]
)

config = {"configurable": {"thread_id": "test_2"}}

# Token counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [9]:
## Run Test
cities = ["Paris", "New Delhi", "Tokyo", "Berlin", "New York", "Cairo", "Dubai"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotets in {city}")]},
        config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: {tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: 255 tokens, 8 messages
[HumanMessage(content='Find hotets in Paris', additional_kwargs={}, response_metadata={}, id='6ef7d720-bbf4-44cb-b545-230d5b162e89'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 59, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_7b8553d7fb', 'id': 'chatcmpl-Dvh6zhUvfZTx87tRd9cv6IcBtcABz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f0dca-4550-7fe2-83df-ef9041cbede5-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_2eEBO4AaupBKaCH887D5gry9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 59,

In [11]:
### Fraction based summarization
@tool
def search_hotels(city: str) -> str:
    """Search hotels in a particular city - returns long response inorder to use more token."""
    return f"""
        Hotels in {city}:
        1. Swastik Grand - 5 Star - spa, pool, gym
        2. Arihant Palace - 5 Star - pool, multicuisine, bar
        3. Navneeta - 3 Star - pool, reception
    """

agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005), # 0.5% = 640 token
            keep=("fraction", 0.002) # 0.2% = 256 token
        ),
    ]
)

config = {"configurable": {"thread_id": "test_2"}}

# Token counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

## Run Test
cities = ["Paris", "New Delhi", "Tokyo", "Berlin", "New York", "Cairo", "Dubai"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotets in {city}")]},
        config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000 # gpt-40-mini context
    print(f"{city}: {fraction} fractions, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: 0.000984375 fractions, 4 messages
[HumanMessage(content='Find hotets in Paris', additional_kwargs={}, response_metadata={}, id='a89f1a6d-f14e-45fc-aa26-9c988f4990e9'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 59, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_7b8553d7fb', 'id': 'chatcmpl-DvhEUl8Z3zTIr2QKMjj8FWxslO8Ce', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f0dd1-625d-7a92-8c39-ea95ec5fa327-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_2prE62rXLE1nRgW3QKnSAEuv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_t

#### Human in the loop Middleware

In [13]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

@tool
def read_email_tool(email_id: str) -> str:
    """Mock Function to read an email by it's ID"""
    return f"Email Content for ID: {email_id}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with the subject {subject} and the body {body}"

agent = create_agent(
    model="gpt-4o",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False
            }
        )
    ]
)


In [14]:
config = {"configurable": {"thread_id": "test_approve"}}

# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with the subject Hello Testing and the body Hello How are you, how it's like in NY?")]},
    config
)

In [ ]:
## here we will not get the direct response but an interrupt that asks for human intervention
result

{'messages': [HumanMessage(content="Send email to john@test.com with the subject Hello Testing and the body Hello How are you, how it's like in NY?", additional_kwargs={}, response_metadata={}, id='668f6354-5c44-4752-a5ca-42f7f444df00'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 104, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a77e540149', 'id': 'chatcmpl-Dvi2RNLx7HGnqSsyTZCSam2d4KoUb', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f0e00-a128-7e12-9c41-a17f9cd52c04-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello Testing', 'body

In [17]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused Approving...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config
    )

    print(f"Result: {result['messages'][-1].content}")

⏸️ Paused Approving...
Result: The email has been sent to john@test.com with the subject "Hello Testing" and the body "Hello How are you, how it's like in NY?".


In [ ]:
## Now the result variable have the final response of the LLM
result

In [18]:
## Now if the human want to reject the request
from langchain.agents.middleware import HumanInTheLoopMiddleware

@tool
def read_email_tool(email_id: str) -> str:
    """Mock Function to read an email by it's ID"""
    return f"Email Content for ID: {email_id}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with the subject {subject} and the body {body}"

agent = create_agent(
    model="gpt-4o",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False
            }
        )
    ]
)


In [19]:
config = {"configurable": {"thread_id": "test_reject"}}

# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with the subject Hello Testing and the body Hello How are you, how it's like in NY?")]},
    config
)

In [20]:
from langgraph.types import Command
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused Approving...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config
    )

    print(f"Result: {result['messages'][-1].content}")

⏸️ Paused Approving...
Result: It seems that the email wasn't sent. If you need any changes or would like to try again, please let me know!


In [21]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with the subject Hello Testing and the body Hello How are you, how it's like in NY?", additional_kwargs={}, response_metadata={}, id='66573d67-20c2-4e19-9e13-18426c3d984e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 104, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a77e540149', 'id': 'chatcmpl-DviCTXRzfzZ9cEtv5yZXc6nYsHaL9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f0e0a-2378-7e80-930c-ad4eff35e3cd-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello Testing', 'body

In [26]:
### If the human wants to edit the email
@tool
def read_email_tool(email_id: str) -> str:
    """Mock Function to read an email by it's ID"""
    return f"Email Content for ID: {email_id}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with the subject {subject} and the body {body}"

agent = create_agent(
    model="gpt-4o",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False
            }
        )
    ]
)

In [27]:
config = {"configurable": {"thread_id": "test_reject"}}

# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with the subject Hello Testing and the body Hello How are you, how it's like in NY?")]},
    config
)

In [28]:
from langgraph.types import Command
# Step 2: Edit
if "__interrupt__" in result:
    print("⏸️ Paused Approving...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool", # Tool Name
                            "args": { # Args of that tool
                                "recipient": "Raju@test.com",
                                "subject": "First one was the wrong one",
                                "body": "This one is the correct one"
                            }
                        }
                    }

                ]
            }
        ),
        config
    )

    print(f"Result: {result['messages'][-1].content}")

⏸️ Paused Approving...
Result: It seems there was a mix-up with the email address. The email was sent to "Raju@test.com" instead of "john@test.com". Could you please confirm the correct recipient's email address for me to send it again?


In [29]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with the subject Hello Testing and the body Hello How are you, how it's like in NY?", additional_kwargs={}, response_metadata={}, id='4648a4e8-1aa9-4718-bef5-3ffa40e584ef'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 104, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a77e540149', 'id': 'chatcmpl-DviIPMXk11QsT5M2LrMSw09zVQ3mw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f0e0f-be93-7b12-92cc-65d542a85a27-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'Raju@test.com', 'subject': '